# CRG + ADD merge

Merge cleaned `V_CRG_STUDENT_COURSE` rows with one ADD semester snapshot row per `student_id`, `degree_id`, and `part_id`.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

JOIN_KEYS = ["student_id", "degree_id", "part_id"]


def find_project_root(start=None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing pyproject.toml and data/.")


PROJECT_ROOT = find_project_root()
PREPROCESSED_DIR = PROJECT_ROOT / "data" / "preprocessed"

CRG_PATH = PREPROCESSED_DIR / "V_CRG_STUDENT_COURSE" / "clean_v_crg_student_course.parquet"
ADD_PATH = PREPROCESSED_DIR / "V_ADD_STUDENT_DEGREE_STATUS" / "clean_v_add_student_degree_status.parquet"

OUTPUT_DIR = PREPROCESSED_DIR / "merge"
OUTPUT_PATH = OUTPUT_DIR / "merge_crg_add.parquet"
UNMATCHED_PATH = OUTPUT_DIR / "merge_crg_add_unmatched_add_snapshot.csv"

for path in [CRG_PATH, ADD_PATH]:
    assert path.exists(), f"Missing input file: {path}"

print("Project root:", PROJECT_ROOT)
print("CRG input:", CRG_PATH)
print("ADD input:", ADD_PATH)
print("Merged output:", OUTPUT_PATH)
print("Unmatched audit output:", UNMATCHED_PATH)

In [ ]:
df_crg = pd.read_parquet(CRG_PATH)
df_add = pd.read_parquet(ADD_PATH)

print("CRG shape:", df_crg.shape)
print("ADD shape:", df_add.shape)

missing_crg_cols = [col for col in JOIN_KEYS if col not in df_crg.columns]
missing_add_cols = [col for col in JOIN_KEYS if col not in df_add.columns]

assert not missing_crg_cols, f"CRG is missing join columns: {missing_crg_cols}"
assert not missing_add_cols, f"ADD is missing join columns: {missing_add_cols}"

print("\nCRG join-key null counts:")
print(df_crg[JOIN_KEYS].isna().sum())

print("\nADD join-key null counts:")
print(df_add[JOIN_KEYS].isna().sum())

print("\nCRG join-key dtypes:")
print(df_crg[JOIN_KEYS].dtypes)

print("\nADD join-key dtypes:")
print(df_add[JOIN_KEYS].dtypes)

## Select ADD snapshot columns

These are semester-start features plus audit fields. CRG remains the row-grain table, so ADD must be unique on the join keys before merging.

In [ ]:
ADD_SNAPSHOT_COLS = [
    "student_status_id",
    "student_id",
    "degree_id",
    "part_id",
    "start_agpa_points",
    "start_agpa_percent",
    "reg_total_semesters",
    "start_level_id",
    "start_level_name_pl",
    "start_part_id",
    "finish_part_id",
    "finish_status",
]

missing_snapshot_cols = [col for col in ADD_SNAPSHOT_COLS if col not in df_add.columns]
assert not missing_snapshot_cols, f"ADD is missing snapshot columns: {missing_snapshot_cols}"

df_add_snapshot = df_add[ADD_SNAPSHOT_COLS].copy()

overlap_cols = [col for col in df_add_snapshot.columns if col in df_crg.columns and col not in JOIN_KEYS]
print("ADD snapshot rows:", len(df_add_snapshot))
print("ADD unique JOIN_KEYS:", df_add_snapshot[JOIN_KEYS].drop_duplicates().shape[0])
print("ADD duplicated rows on JOIN_KEYS:", df_add_snapshot.duplicated(JOIN_KEYS, keep=False).sum())
print("ADD columns that will receive '_add_snapshot' suffix:", overlap_cols)

if df_add_snapshot.duplicated(JOIN_KEYS).any():
    display(
        df_add_snapshot.loc[df_add_snapshot.duplicated(JOIN_KEYS, keep=False)]
        .sort_values(JOIN_KEYS)
        .head(100)
    )

assert df_add_snapshot.duplicated(JOIN_KEYS).sum() == 0, "ADD is not unique on JOIN_KEYS."

## Pre-merge validation

In [ ]:
print("=" * 80)
print("PRE-MERGE VALIDATION")
print("=" * 80)

print("CRG rows before merge:", len(df_crg))
print("ADD snapshot rows:", len(df_add_snapshot))
print("ADD unique JOIN_KEYS:", df_add_snapshot[JOIN_KEYS].drop_duplicates().shape[0])

df_merge_test = df_crg.merge(
    df_add_snapshot,
    on=JOIN_KEYS,
    how="left",
    validate="many_to_one",
    indicator=True,
    suffixes=("", "_add_snapshot"),
)

print("\nRows after merge:", len(df_merge_test))
print("Row count changed:", len(df_merge_test) - len(df_crg))

assert len(df_merge_test) == len(df_crg), "Row count changed after merge. Investigate duplicate ADD keys."

print("\nMerge result:")
print(df_merge_test["_merge"].value_counts(dropna=False))

missing_snapshot_mask = df_merge_test["_merge"].eq("left_only")

print("\nMissing ADD snapshot rows:", missing_snapshot_mask.sum())
print("Missing ADD snapshot ratio:", missing_snapshot_mask.mean())

unmatched_snapshot_report = df_merge_test.loc[missing_snapshot_mask].copy()

print("\nUnmatched rows by part_id:")
display(unmatched_snapshot_report["part_id"].value_counts(dropna=False).head(30))

print("\nUnmatched rows by degree_id:")
display(unmatched_snapshot_report["degree_id"].value_counts(dropna=False).head(30))

if "register_status" in unmatched_snapshot_report.columns:
    print("\nUnmatched rows by register_status:")
    display(unmatched_snapshot_report["register_status"].value_counts(dropna=False))

finish_status_col = "finish_status" if "finish_status" in unmatched_snapshot_report.columns else "finish_status_crg"
if finish_status_col in unmatched_snapshot_report.columns:
    print("\nUnmatched rows by CRG finish_status:")
    display(unmatched_snapshot_report[finish_status_col].value_counts(dropna=False))

sample_cols = [
    "student_course_id",
    "student_id",
    "degree_id",
    "part_id",
    "course_id",
    "grade_id",
    "final_mark",
    "points",
    finish_status_col,
    "register_status",
]
sample_cols = [col for col in sample_cols if col in unmatched_snapshot_report.columns]

print("\nUnmatched sample:")
display(unmatched_snapshot_report[sample_cols].head(100))

## Save merged table

In [ ]:
df_crg_add = df_merge_test.copy()
df_crg_add["has_add_snapshot"] = df_crg_add["_merge"].eq("both")
df_crg_add = df_crg_add.drop(columns=["_merge"])

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df_crg_add.to_parquet(OUTPUT_PATH, index=False)

unmatched_export_cols = [
    "student_course_id",
    "student_id",
    "degree_id",
    "part_id",
    "course_id",
    "grade_id",
    "final_mark",
    "points",
    finish_status_col,
    "register_status",
]
unmatched_export_cols = [col for col in unmatched_export_cols if col in unmatched_snapshot_report.columns]
unmatched_snapshot_report[unmatched_export_cols].to_csv(UNMATCHED_PATH, index=False)

print("Saved merged parquet:", OUTPUT_PATH)
print("Saved unmatched audit CSV:", UNMATCHED_PATH)
print("Merged shape:", df_crg_add.shape)
print("Rows with ADD snapshot:", int(df_crg_add["has_add_snapshot"].sum()))
print("Rows missing ADD snapshot:", int((~df_crg_add["has_add_snapshot"]).sum()))

In [ ]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

In [ ]:
display(df_crg_add.head())

In [ ]:
df_crg_add.info()

In [ ]:
df_crg_add.columns

In [ ]:
df_acd=pd.read_parquet(r"D:\AI\Real projects\Academic_Advisor\data\preprocessed\V_ACD_DEGREE_COURSE\v_acd_degree_course.parquet")